# SHAP Explanations for TLDC

This notebook explains one prediction from a saved classical model and links exact attribution spans to Week 4 section/page provenance. SHAP is an estimate of model behavior, not legal reasoning or proof. Do not execute attribution until the dataset and model are approved and versioned.

Install optional dependencies with `pip install -r requirements-xai.txt`. This notebook uses the trained artifact path `models/classical_ml/document_classifier.joblib`; the Week 5 baseline script does not create that artifact automatically.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / "models/classical_ml/document_classifier.joblib"
TRAIN_PATH = ROOT / "data/train/tldc_train.jsonl"
TEST_PATH = ROOT / "data/test/tldc_test.jsonl"
SEGMENTS_PATH = ROOT / "annotations/document_segmentation/segmented_documents.jsonl"
OUTPUT_PATH = ROOT / "xai/shap/shap_results.jsonl"
RUN_SHAP = False


def read_jsonl(path):
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as source:
        return [json.loads(line) for line in source if line.strip()]


training_rows = read_jsonl(TRAIN_PATH)
test_rows = [row for row in read_jsonl(TEST_PATH) if row.get("task") == "document_classification"]
if not RUN_SHAP:
    print("SHAP is disabled; set RUN_SHAP=True only after model and reviewed test data are available.")
elif not MODEL_PATH.exists() or not training_rows or not test_rows:
    print("Skip SHAP: saved model, training background, and held-out reviewed example are required.")
else:
    import joblib
    from src.explainability.shap_explainer import explain_text
    from src.explainability.evidence_linker import link_attributions

    model = joblib.load(MODEL_PATH)
    source_documents = {row["document_id"]: row for row in read_jsonl(SEGMENTS_PATH)}
    example = test_rows[0]
    source_document = source_documents.get(example["document_id"], {})
    explanation = explain_text(
        model,
        example["text"],
        [row["text"] for row in training_rows if row.get("text")],
        model_version="tldc_baseline_v1",
    )
    record = link_attributions(
        example["document_id"], example["text"], source_document.get("sections", []),
        explanation["attributions"], explanation["predicted_label"], explanation["model_version"],
        explanation["method"], score=explanation["score"],
    )
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    with OUTPUT_PATH.open("a", encoding="utf-8") as output:
        output.write(json.dumps(record, ensure_ascii=False) + "\n")
    record

SHAP is disabled; set RUN_SHAP=True only after model and reviewed test data are available.


In [2]:
# Optional global view: mean absolute SHAP contribution by token across reviewed examples.
if RUN_SHAP and "model" in globals() and test_rows:
    from collections import defaultdict
    import pandas as pd
    from src.explainability.shap_explainer import explain_text

    contribution_totals = defaultdict(float)
    contribution_counts = defaultdict(int)
    for row in test_rows[:20]:
        local = explain_text(model, row["text"], [item["text"] for item in training_rows if item.get("text")], "tldc_baseline_v1")
        for item in local["attributions"]:
            feature = item.get("feature", row["text"][item["start"] : item["end"]])
            contribution_totals[feature] += abs(item["score"])
            contribution_counts[feature] += 1
    global_summary = pd.DataFrame([
        {"feature": feature, "mean_absolute_shap": contribution_totals[feature] / contribution_counts[feature]}
        for feature in contribution_totals
    ]).sort_values("mean_absolute_shap", ascending=False)
    display(global_summary.head(30))
else:
    print("Global SHAP aggregation awaits RUN_SHAP=True and eligible reviewed examples.")

Global SHAP aggregation awaits RUN_SHAP=True and eligible reviewed examples.
